# Multiple labels per clip: `multiselect`

A `multiselect` lets you tag one clip (or one box) with several labels: *a cat, a generator and an owl*. Pick a label from the dropdown and it becomes a tag under it; click a tag's **×** to remove it.

How the tags are saved is set by `output`:

| Option | Config | Saved as |
|---|---|---|
| **A. names** | `items: {path, value}` | one column holding a list of names: `["Owl", "Cat"]` |
| **D. ids** | `items: {path, value, id}` | one column holding a list of ids from the items file: `[121, 150]` (tags still show the names) |
| **B. one-hot** | `output: {format: columns}` | one column per item: `has_owl=1, has_cat=0, …` |

In a CSV the list is JSON text; parquet and JSON Lines outputs store a real list. Each form below is a plain dict, so you can read it here and pass it as `form_config`.

In [ ]:
import json

import pandas as pd

from jupyter_bioacoustic import BioacousticAnnotator

DATA = 'data/detections.N-dse.csv'
SPECIES = 'data/species_counts.N-dse.csv'     # common_name + an integer `index` column (class id)

# settings shared by every example below
COMMON = dict(
    data=DATA,
    data_index_column='id',
    audio='audio_url',
    display_columns=['common_name', 'confidence', 'start_time'],
    info_card_title='[[common_name]]',
    duplicate_entries=False,
)


def read_tags(ba, column):
    """The saved rows, with ``column`` turned from JSON text (CSV) back into a list."""
    out = ba.output(force=True)
    if out is None:
        print('Nothing saved yet: tag a clip above and press Submit.')
        return None
    out[column] = out[column].map(json.loads)
    return out

## A. A list of names

The tags are names from the species file. `custom_value` allows tags that aren't in the file (type one in the filter box and press **+ Add**), and `not_available` adds a *Nothing identifiable* tag that replaces any others. `source_value` starts each clip with the detector's label already tagged, so you only remove wrong tags and add missing ones.

To save a delimited string instead of a list, add `'output': {'delimiter': ';'}`.

In [ ]:
FORM_A = {
    'title': {'value': 'WHAT CAN YOU HEAR?', 'progress_tracker': True},
    'form': [
        {'multiselect': {
            'label': 'species heard',
            'column': 'species',
            'required': True,                  # at least one tag (also: 'min', 'max')
            'source_value': 'common_name',     # start with the detector's label
            'items': {
                'path': SPECIES,
                'value': 'common_name',
                'filter_box': True,
                'custom_value': True,
                'not_available': {'label': 'Nothing identifiable', 'value': 'none'},
            },
            # 'output': {'format': 'list'} is the default
        }},
        {'textbox': {'label': 'notes', 'column': 'notes'}},
    ],
}

ba_a = BioacousticAnnotator(**COMMON, form_config=FORM_A, output='outputs/multilabel-a-names.csv')
print(ba_a.validate())
ba_a.open()

In [ ]:
a = read_tags(ba_a, 'species')
if a is not None:
    display(a[['id', 'species', 'notes']].tail())
    # one row per tag, or one-hot columns, when you need them
    display(a['species'].explode().value_counts().head(10))
    display(a['species'].str.join('|').str.get_dummies().tail())

## D. A list of class ids

Add `id` to `items` to save another column of the items file instead of the name, here the integer class `index`. The tags still show `common_name`. Integer ids are saved as numbers. Without `id` this is exactly option A.

In [ ]:
FORM_D = {
    'title': {'value': 'WHAT CAN YOU HEAR?', 'progress_tracker': True},
    'form': [
        {'multiselect': {
            'label': 'species heard',
            'column': 'species_ids',
            'required': True,
            'items': {
                'path': SPECIES,
                'value': 'common_name',        # shown on the tags
                'id': 'index',                 # saved
                'filter_box': True,
            },
        }},
    ],
}

ba_d = BioacousticAnnotator(**COMMON, form_config=FORM_D, output='outputs/multilabel-d-ids.csv')
print(ba_d.validate())
ba_d.open()

In [ ]:
d = read_tags(ba_d, 'species_ids')
if d is not None:
    names = pd.read_csv(SPECIES).set_index('index')['common_name']
    d['species'] = d['species_ids'].map(lambda ids: [names[i] for i in ids])
    display(d[['id', 'species_ids', 'species']].tail())

## B. One column per label (one-hot)

For a short, fixed list of labels, `format: columns` writes a column per item (`column_prefix` + the item's value) holding `true_value` or `false_value`. It needs a closed list, so `custom_value` isn't allowed. For thousands of classes (e.g. BirdNET's), use A or D instead.

In [ ]:
SOURCES = [
    {'label': 'Bird', 'value': 'bird'},
    {'label': 'Insect', 'value': 'insect'},
    {'label': 'Engine / generator', 'value': 'engine'},
    {'label': 'Dog', 'value': 'dog'},
    {'label': 'Human voice', 'value': 'voice'},
    {'label': 'Rain', 'value': 'rain'},
    {'label': 'Wind', 'value': 'wind'},
]

FORM_B = {
    'title': {'value': 'SOUND SOURCES', 'progress_tracker': True},
    'form': [
        {'multiselect': {
            'label': 'sources present',
            'column': 'source',
            'required': True,
            'items': SOURCES,
            'output': {
                'format': 'columns',
                'column_prefix': 'has_',       # has_bird, has_insect, ...
                'true_value': 1,
                'false_value': 0,
            },
        }},
    ],
}

ba_b = BioacousticAnnotator(**COMMON, form_config=FORM_B, output='outputs/multilabel-b-onehot.csv')
print(ba_b.validate())
ba_b.open()

In [ ]:
b = ba_b.output(force=True)
if b is None:
    print('Nothing saved yet: tag a clip above and press Submit.')
else:
    display(b[['id'] + [c for c in b.columns if c.startswith('has_')]].tail())

## Tags per box

A `multiselect` also works in the per-box form of the `multibox` tool: each box gets its own tags and is saved as its own row.

In [ ]:
FORM_BOXES = {
    'title': {'value': 'TAG EACH BOX'},
    'annotation': {'tools': ['multibox'], 'form': 'box_tags'},
    'dynamic_forms': {
        'box_tags': [
            {'multiselect': {
                'label': 'in this box',
                'column': 'species',
                'required': True,
                'items': {'path': SPECIES, 'value': 'common_name', 'filter_box': True},
            }},
        ],
    },
}

ba_boxes = BioacousticAnnotator(**COMMON, form_config=FORM_BOXES, output='outputs/multilabel-boxes.csv')
print(ba_boxes.validate())
ba_boxes.open()